# DiveLab

## Notebook 01 — Depth and Pressure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/01_Pressure.ipynb)

**Guiding question:** How does pressure change as a diver descends?

*Every dynamic model begins with its environment.*

## Learning objectives

By the end of this lab, you will be able to:

- explain why pressure increases with depth;
- distinguish absolute pressure from gauge pressure;
- calculate pressure at a given depth;
- implement the pressure–depth relationship in Python;
- visualize pressure as a function of depth;
- explain why **relative pressure changes are largest near the surface**.

## Physical intuition

At the surface, a diver is already exposed to atmospheric pressure.

When the diver descends, the water above adds hydrostatic pressure. Therefore, the deeper the diver goes, the greater the ambient pressure.

A useful diving approximation in seawater is:

- 0 m → about 1 bar
- 10 m → about 2 bar
- 20 m → about 3 bar
- 30 m → about 4 bar

The key point is that each additional 10 m adds roughly the same **absolute** pressure, but not the same **relative** pressure change.

## Mathematical model

For a fluid with approximately constant density,

$$
\frac{dP}{dz}=\rho g
$$

where:

- $P$ is pressure;
- $z$ is depth, taken as positive downward;
- $\rho$ is water density;
- $g$ is gravitational acceleration.

Integrating with depth gives

$$
P(z)=P_0+\rho gz
$$

where $P_0$ is atmospheric pressure at the surface.

This is the **absolute pressure** experienced by the diver.

## Absolute and gauge pressure

Absolute pressure includes atmospheric pressure:

$$
P_{\mathrm{abs}} = P_0 + \rho gz
$$

Gauge pressure measures only the contribution of the water:

$$
P_{\mathrm{gauge}} = \rho gz
$$

At the surface, gauge pressure is zero while absolute pressure is approximately 1 bar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Physical constants
rho = 1025.0       # seawater density [kg/m^3]
g = 9.80665        # gravitational acceleration [m/s^2]
P0 = 101325.0      # atmospheric pressure [Pa]

## First calculation

Let's calculate the absolute pressure at 10 m using the equation directly.

In [ ]:
depth_m = 10

pressure_pa = P0 + rho * g * depth_m
pressure_bar_10m = pressure_pa / 100_000

print(f"Pressure at {depth_m} m = {pressure_bar_10m:.3f} bar")

## Turn the model into reusable Python functions

We now wrap the equations in functions. The same function will work with both a single depth and a NumPy array of depths.

In [ ]:
def pressure_at_depth(depth_m):
    """Return absolute pressure in pascals at the given depth in seawater."""
    return P0 + rho * g * depth_m


def pressure_bar(depth_m):
    """Return absolute pressure in bar at the given depth in seawater."""
    return pressure_at_depth(depth_m) / 100_000


def gauge_pressure_bar(depth_m):
    """Return gauge pressure in bar at the given depth in seawater."""
    return (rho * g * depth_m) / 100_000

## Experiment — pressure at typical diving depths

In [ ]:
for depth_m in [0, 10, 20, 30, 40]:
    print(f"{depth_m:2d} m → {pressure_bar(depth_m):.2f} bar absolute")

The familiar rule “1 bar every 10 m” is an approximation. Our result is slightly different because the model uses realistic values for seawater density, gravitational acceleration, and atmospheric pressure.

## Visualize pressure versus depth

In [ ]:
depth = np.linspace(0, 40, 200)
pressure = pressure_bar(depth)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(depth, pressure)
plt.xlabel("Depth [m]")
plt.ylabel("Absolute pressure [bar]")
plt.title("Ambient pressure vs depth")
plt.grid(True)
plt.show()

## The first meters matter most

Pressure increases approximately **linearly** with depth: each additional 10 m adds about 1 bar.

But the **relative pressure change** is not constant.

Using the simple diving approximation:

- 0 → 10 m: 1 → 2 bar = **+100%**
- 10 → 20 m: 2 → 3 bar = **+50%**
- 20 → 30 m: 3 → 4 bar = **+33%**
- 30 → 40 m: 4 → 5 bar = **+25%**

So the same 10 m descent produces roughly the same absolute pressure increase, but a progressively smaller relative increase.

> **The largest relative pressure changes occur near the surface.**

In [ ]:
depths = np.array([0, 10, 20, 30, 40])
pressures = pressure_bar(depths)

relative_increase = np.diff(pressures) / pressures[:-1] * 100

for start, end, change in zip(depths[:-1], depths[1:], relative_increase):
    print(f"{start:2d} → {end:2d} m: {change:.1f}%")

## Relative rate of pressure change

The absolute pressure gradient is constant:

$$
\frac{dP}{dz}=\rho g
$$

But the **relative** rate of change is

$$
\frac{1}{P}\frac{dP}{dz}
=
\frac{\rho g}{P_0+\rho gz}
$$

Because the denominator grows with depth, the relative rate decreases as the diver goes deeper.

In [ ]:
relative_rate = (rho * g) / pressure_at_depth(depth)   # per meter

plt.figure(figsize=(8, 5))
plt.plot(depth, relative_rate * 100)
plt.xlabel("Depth [m]")
plt.ylabel("Relative pressure change [% per m]")
plt.title("Relative pressure change is greatest near the surface")
plt.grid(True)
plt.show()

## Interpretation

The pressure–depth relationship is linear, but its **relative effect** is strongly depth-dependent.

This is a crucial diving insight: the first meters of descent and the last meters of ascent contain the largest relative pressure changes.

In the next notebook we will connect this result to **Boyle's law** and ask:

> **What does this pressure change imply for a gas volume?**

## Exercises

### 1. Fresh water

Repeat the calculation using fresh water density:

$$
\rho = 1000\ \mathrm{kg/m^3}
$$

How much does the pressure at 40 m change compared with seawater?

### 2. Find a depth

At what depth is the absolute pressure approximately 6 bar?

First estimate it mentally using the diving approximation, then calculate it with the model.

### 3. Gauge pressure

Use `gauge_pressure_bar(depth_m)` and compare absolute and gauge pressure at 0 m, 10 m, 20 m, and 30 m.

### 4. Relative change

Compute the relative pressure increase for 5 m intervals from 0 to 40 m. Where is it largest?

## Challenge — recover the pressure gradient numerically

Generate pressure values over a range of depths and estimate

$$
\frac{dP}{dz}
$$

numerically with NumPy.

Compare the numerical result with the theoretical value $\rho g$.

In [ ]:
# Starter code for the challenge
z = np.linspace(0, 40, 401)
P = pressure_at_depth(z)

dP_dz_numeric = np.gradient(P, z)

print(f"Theoretical rho*g: {rho * g:.2f} Pa/m")
print(f"Mean numerical dP/dz: {dP_dz_numeric.mean():.2f} Pa/m")

## Summary

In this lab we learned that:

- water pressure increases approximately linearly with depth;
- absolute pressure includes atmospheric pressure;
- gauge pressure measures pressure relative to atmospheric pressure;
- in seawater, ambient pressure increases by roughly 1 bar every 10 m;
- **relative pressure changes are greatest near the surface**.

Next: **Boyle's law and gas volume**.